**ANÁLISE DE DADOS - MINI PROJETO AVALIATIVO - BASE DE DADOS VAREJO**

**Aluno: Henrique da Silveira** 
**Data: 02-06-2026**

**IMPORTAÇÕES INICIAIS**

In [42]:
#Importação de bibliotecas e exibição mais legível

import pandas as pd
import numpy as np
import sys
import warnings


#Importação das funções utils

sys.path.append('..')
from utils.funcoes_varejo import *


# Configurações para deixar a exibição mais legível
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)   # Mostrar todas as colunas
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 casas decimais

#Importação base de dados
df = pd.read_csv('../data/Base Varejo.csv', sep=';')

#Realiza disgnótico inicial dos dados
diagnostico(df,"Dados Varejo")

  📊 DIAGNÓSTICO: Dados Varejo
  Linhas:           830,000
  Colunas:          14
  Linhas duplicadas:96,553

  Coluna           | Tipo       | Nulos | % Nulos
  --------------------------------------------------
  DATA              | str        | 0     | 0.0%
  CO_ID             | int64      | 0     | 0.0%
  CL_ID             | int64      | 0     | 0.0%
  CL_GENERO         | str        | 0     | 0.0%
  CL_EC             | int64      | 0     | 0.0%
  CL_FHL            | int64      | 0     | 0.0%
  CL_SEG            | str        | 0     | 0.0%
  PR_ID             | int64      | 0     | 0.0%
  PR_CAT            | str        | 0     | 0.0%
  PR_NOME           | str        | 0     | 0.0%
  Unnamed: 10       | float64    | 830000 | 100.0%
  Unnamed: 11       | float64    | 830000 | 100.0%
  Unnamed: 12       | float64    | 830000 | 100.0%
  Unnamed: 13       | float64    | 830000 | 100.0%


In [43]:
#Realiza uma cópia do data frame original para realizar a limpeza e tratamento
df_tratamento = df.copy()

In [44]:
#Corrige tipo de dados da coluna data

# Texto → data (dayfirst=True para datas no formato brasileiro DD/MM/YYYY)
df_tratamento['DATA'] = pd.to_datetime(df_tratamento['DATA'],format='%d/%m/%Y', dayfirst=True, errors='coerce')
print(df_tratamento.dtypes)

DATA           datetime64[us]
CO_ID                   int64
CL_ID                   int64
CL_GENERO                 str
CL_EC                   int64
CL_FHL                  int64
CL_SEG                    str
PR_ID                   int64
PR_CAT                    str
PR_NOME                   str
Unnamed: 10           float64
Unnamed: 11           float64
Unnamed: 12           float64
Unnamed: 13           float64
dtype: object


In [45]:
# Limpar colunas vazia encontradas
df_tratamento = df_tratamento.dropna(axis=1, how='all')

#Exibe para confirmar que as colunas foram deletadas
print(df_tratamento.columns.tolist())

['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME']


** TRANSFORMÇÕES / LIMPEZA DE DADOS**

In [46]:
#Substitui valores nulos que não são dectataveis para detectaveis com pandas
df_tratamento= df_tratamento.replace({'NULL': np.nan, 'N/A': np.nan, '': np.nan, '#N/D': np.nan})

#Verifica se há espaços vazios que também não são nativamente detectáveis
for col in df_tratamento.columns:
    qtd = (df_tratamento[col].astype(str).str.strip() == '').sum()
    if qtd > 0:
        #Mostra que encontrou e a quantidade
        print(f'{col}: {qtd} VALORES VAZIOS ENCONTRADOS')
        
        #Realiza o tratamento destes valores caso sejam encontrados
        df_tratamento[col] = df_tratamento[col].replace(r'^\s*$', np.nan, regex=True)        
    else:        
        print(f'{col}: {qtd} Não há valores com espaços vazios')

DATA: 0 Não há valores com espaços vazios
CO_ID: 0 Não há valores com espaços vazios
CL_ID: 0 Não há valores com espaços vazios
CL_GENERO: 0 Não há valores com espaços vazios
CL_EC: 0 Não há valores com espaços vazios
CL_FHL: 0 Não há valores com espaços vazios
CL_SEG: 0 Não há valores com espaços vazios
PR_ID: 0 Não há valores com espaços vazios
PR_CAT: 0 Não há valores com espaços vazios
PR_NOME: 0 Não há valores com espaços vazios


In [47]:
#Depois das substituições, realiza soma dos nulos detectáveis 
qtd_nulos = (df_tratamento.isnull().sum())

#Verifica se constam nulos após replace e se há dados que precisam de tratamento
for col in df_tratamento.columns:
     if qtd_nulos[col] > 0:
          print(f"{col}: {qtd_nulos[col]} VALORES NULOS ENCONTRADOS!")          
     else:
          print(f"{col}: SEM NULOS PARA TRATAR")

DATA: SEM NULOS PARA TRATAR
CO_ID: SEM NULOS PARA TRATAR
CL_ID: SEM NULOS PARA TRATAR
CL_GENERO: SEM NULOS PARA TRATAR
CL_EC: SEM NULOS PARA TRATAR
CL_FHL: SEM NULOS PARA TRATAR
CL_SEG: SEM NULOS PARA TRATAR
PR_ID: SEM NULOS PARA TRATAR
PR_CAT: 3650 VALORES NULOS ENCONTRADOS!
PR_NOME: 3650 VALORES NULOS ENCONTRADOS!


**TRATAMENTO DE VALORES NULOS / AUSENTES**

In [48]:
#TRATAMENTO PRODUTOS SEM NOME
df_tratamento['PR_NOME'] = df_tratamento["PR_CAT"].fillna('NÃO INFORMADO')

#TRATAMENTO DE PRODUTOS SEM CATEGORIA
df_tratamento['PR_CAT'] = df_tratamento["PR_CAT"].fillna('SEM CATEGORIA')

AGRUPAMENTOS

In [49]:
#PADRÃO 1: Categoria de Produto por Gênero
print("\n--- PADRÃO 1: Relação de Produto por Gênero ---")

# Agrupando por Gênero e Categoria do Produto
consumo_genero = df_tratamento.groupby(['PR_CAT', 'CL_GENERO']).size().reset_index(name='Total')
print(consumo_genero.to_string(index=False))


--- PADRÃO 1: Relação de Produto por Gênero ---
       PR_CAT CL_GENERO  Total
   ACESSORIOS         F   7700
   ACESSORIOS         M   6857
    ALIMENTOS         F 226575
    ALIMENTOS         M 208192
      BEBIDAS         F  22345
      BEBIDAS         M  20954
      HIGIENE         F  80991
      HIGIENE         M  74583
      LIMPEZA         F  76298
      LIMPEZA         M  69456
          PET         F  16749
          PET         M  15650
SEM CATEGORIA         F   1918
SEM CATEGORIA         M   1732


In [50]:
#PADRÃO 2: Estrutura Familiar vs. Preferência de Categoria

print("\n--- PADRÃO 2: Estrutura Familiar vs. Categorias ---")

# Agrupamento combinando 3 variáveis: Estado Civil, Filhos e Categoria do Produto
padrao_familiar = df_tratamento.groupby(['CL_EC', 'CL_FHL', 'PR_CAT']).size().reset_index(name='Quantidade_Vendida')

# Combinações mais frequentes desse agrupamento (TOP 15)
print(padrao_familiar.sort_values(by='Quantidade_Vendida', ascending=False).head(15))


--- PADRÃO 2: Estrutura Familiar vs. Categorias ---
     CL_EC  CL_FHL     PR_CAT  Quantidade_Vendida
106      4       0  ALIMENTOS               61324
71       3       0  ALIMENTOS               59589
1        1       0  ALIMENTOS               56582
36       2       0  ALIMENTOS               50576
108      4       0    HIGIENE               21979
73       3       0    HIGIENE               21351
109      4       0    LIMPEZA               20594
3        1       0    HIGIENE               20317
74       3       0    LIMPEZA               20220
4        1       0    LIMPEZA               18747
38       2       0    HIGIENE               18009
113      4       1  ALIMENTOS               17219
39       2       0    LIMPEZA               17136
57       2       3  ALIMENTOS               16191
99       3       4  ALIMENTOS               16175


In [51]:
#PADRÃO 3: Consumo por Segmento Econômico e Período
print("\n--- PADRÃO 3: Segmento Econômico vs. Tempo ---")

# Confirma o formato correto da data
df['DATA'] = pd.to_datetime(df['DATA'], dayfirst=True)

# Extraindo o dia da semana 
df['DIA'] = df['DATA'].dt.day_name(locale='pt_BR')

# Agrupa Classe Econômica com o Dia da Semana para ver o fluxo de compras
fluxo_segmento = df.groupby(['CL_SEG', 'DIA'])['CO_ID'].count().unstack(fill_value=0)

#Corrige visualização
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

print("\nVolume de compras por Segmento e Dia da Semana:")
print(fluxo_segmento)


--- PADRÃO 3: Segmento Econômico vs. Tempo ---

Volume de compras por Segmento e Dia da Semana:
DIA     Domingo  Quarta-feira  Quinta-feira  Segunda-feira  Sexta-feira  Sábado  Terça-feira
CL_SEG                                                                                      
A         11016         12354          8237           7539        11452    8192         8946
B         84623         98126         66870          72082        85364   59214        63884
C         34864         43080         30367          32165        35817   26982        28826


**Análise de dados estatisticos - Números de Filhos dos cliente**

In [52]:
# Contagem Geral de dados
print('TOTAL DE REGISTROS:')
print(df_tratamento['CL_FHL'].count())

# Analise de Média, Mediana e Moda relacionadas ao número de filhos dos clientes
print(f"Média de filhos: {df['CL_FHL'].mean():.2f}")   

print(f"Mediana: {df['CL_FHL'].median()}")   

# Retorna o valor mais comum de filhos
print(f"Moda: {df['CL_FHL'].mode()[0]}")  

#Quantidade máxima de filhos informada
print(f"Máximo: {df['CL_FHL'].max()}")

#Porcentagem de clientes com vs sem filhos
print(f"Clientes sem filhos: {(df['CL_FHL'] == 0).mean()*100:.2f}%")
print(f"Clientes com filhos: {(df['CL_FHL'] > 0).mean()*100:.2f}%")

TOTAL DE REGISTROS:
830000
Média de filhos: 1.15
Mediana: 0.0
Moda: 0
Máximo: 4
Clientes sem filhos: 52.47%
Clientes com filhos: 47.53%


**SALVAMENTO DO DATAFRAME LIMPO**

In [53]:
# Salva o DataFrame como um arquivo CSV
df_tratamento.to_csv('../df_limpo.csv', index=False, encoding='utf-8-sig', sep=';')
print("Arquivo CSV salvo!")

Arquivo CSV salvo!


Insgights Obtidos:

- 💡 #PADRÃO 1: Em todas as categorias de produtos, as mulheres são as principais compradoras, mostrando que são o públivo de maior interesse e que pode ser trabalhado formas de atrair mais compradores masculinos.

- 💡 #PADRÃO 2: Os clientes das primeiras posições da lista são estruturas familiares sem filhos, portanto, promoções focadas em "ITENS FAMILIARES" ou de grandes volumes podem não ser atrativas.

- 💡 #PADRÃO 3: Em todos os dias da semana, a classe B é o publico que mais compra, portanto deve ser o principal alvo do varejo.
 A classe A costuma ser mais frequente as quartas, verificar que tipo de oferta é oferecidas as quartas e aplica-las em outros dias da semana onde se deseja que esse público seja mais frequente.

- Na análise estatistica relizada sobre os filhos, percebe-se que as familias não são grandes.